### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. They\'re known for mimicking human speech, right? But why do they do that? Maybe it\'s related to their natural behavior.\n\nFirst, I remember that parrots are social animals. In the wild, they live in flocks. So maybe talking is a way they communicate with each other. But how does that translate to mimicking human words? Maybe they use vocalizations to interact with their flock, and when they\'re around humans, they start mimicking our sounds. So it\'s a form of social interaction.\n\nAlso, some parrots can produce a wide range of sounds because of their vocal anatomy. They have a syrinx, which is their voice box, and it\'s different from humans. That might allow them to mimic various sounds, including human speech. So their anatomy plays a role.\n\nAnother angle is intelligence. Parrots are considered intelligent birds. They might learn to talk as a way to 

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location: str)->str:
    """Get the weather at a location"""
    return f"its sunny in {location}"

model_with_tools=model.bind_tools([get_weather])

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x113aa2f90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x113aa3cb0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather at a location', 'parameters': {'p

In [4]:
response = model_with_tools.invoke("what's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there\'s no typo. Everything looks good. Let\'s format the tool call correctly.\n', 'tool_calls': [{'id': 'krwkky09a', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 154, 'total_tokens': 246, 'completion_time': 0.143553805, 'completion_tokens_details': {'reasoning_tokens': 68}, 'prompt_time': 0.006665133, 'prompt_tokens_details': None, 'queue_time': 0.163773867, 'total_time': 0.150218938}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_ru

### Tool Execution Loops

In [8]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass resulta back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)


The weather in Boston is sunny. Let me know if you need more details!


In [9]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there are no typos. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'tz8b34aen', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 153, 'total_tokens': 247, 'completion_time': 0.138670755, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.006097824, 'prompt_tokens_details': None, 'queue_time': 0.056848736, 'total_time': 0.144768579}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finis